In [0]:
print("Accenture_Databricks_Virtue_Agent")

Accenture_Databricks_Virtue_Agent


In [0]:
%pip install langchain-groq pydantic typing pandas

import os, getpass, time
import pandas as pd
from pydantic import BaseModel, Field
from typing import List, Optional
from langchain_groq import ChatGroq

# Step 1: Rules for the AI (As per Hackathon PDF)
class MedicalKnowledge(BaseModel):
    facility_name: str = Field(description="Name of the hospital")
    specialty: str = Field(description="Must be one of: internalMedicine, familyMedicine, pediatrics, cardiology, generalSurgery, emergencyMedicine, gynecologyAndObstetrics, orthopedicSurgery, dentistry, ophthalmology")
    procedures: List[str] = Field(description="Clinical services (e.g., 'Offers dialysis'). Return [] if none.")
    equipment: List[str] = Field(description="Machines like CT/X-ray. Return [] if none.")
    capability: List[str] = Field(description="ICU/Trauma levels. NO contact/pricing. Return [] if none.")
    is_suspicious: bool = Field(description="Strictly True or False (Boolean).")
    suspicion_reason: str = Field(default="",description="Reason for flagging if suspicious")

print("✅ Step 1: Schema Ready!")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
✅ Step 1: Schema Ready!


In [0]:
%sql
SHOW TABLES IN default


database,tableName,isTemporary
default,virtue_foundation_ghana_v_0_3_sheet_1,false


In [0]:
# 'default.ghana_health_data' ki jagah apna table name likhna
df = spark.table("workspace.default.virtue_foundation_ghana_v_0_3_sheet_1") 

# Total rows check karne ke liye
print(f"Total Records in Dataset: {df.count()}")

# Saara data dekhne ke liye (Scrollable view)
display(df)

Total Records in Dataset: 995


source_url,name,pk_unique_id,mongo DB,specialties,procedure,equipment,capability,organization_type,content_table_id,phone_numbers,email,websites,officialWebsite,yearEstablished,acceptsVolunteers,facebookLink,twitterLink,linkedinLink,instagramLink,logo,address_line1,address_line2,address_line3,address_city,address_stateOrRegion,address_zipOrPostcode,address_country,address_countryCode,countries,missionStatement,missionStatementLink,organizationDescription,facilityTypeId,operatorTypeId,affiliationTypeIds,description,area,numberDoctors,capacity,unique_id
https://www.linkedin.com/company/waaf/,"109/No 1 Bekwai Rd (Near Mexico Hotel) Takoradi, Ghana",1,62aa51490990af00169ab9ed,"[""infectiousDiseases"",""maternalFetalMedicineOrPerinatology"",""publicHealth"",""socialAndBehavioralSciences"",""hospiceAndPalliativeInternalMedicine"",""globalHealthAndInternationalHealth""]",null,null,"[""Has a location at 109/No 1 Bekwai Rd (Near Mexico Hotel) Takoradi, Ghana"",""Specialties include HIV, AIDS, PMTCT, Behavior Change Communication, HIV Testing and Counseling, Community Outreach, Stigma Reduction, Hospice / Home Based Care, global health, and public health""]",facility,a77400f4-6203-4b0d-84ad-fd143bd768e3,"[""+233249354576"",""+233203928883""]",null,"[""waafweb.org""]",waafweb.org,null,null,null,null,null,null,null,109/No 1 Bekwai Rd (Near Mexico Hotel),null,null,Takoradi,null,null,Ghana,GH,null,null,null,null,clinic,null,null,"WAAF is committed to battling HIV/AIDS, TB, and other conditions through grassroot initiatives throughout communities. The main objectives for WAAF are to deliver prevention, behavioral, care and support services for the general population but with emphasis on vulnerable populations. Our ultimate goal is a future where HIV and AIDS is no longer an epidemic, and people are no longer stigmatized. The organization’s vision and mission guide for daily and long-term operations, as detailed in the organization’s strategic plan document.",null,null,null,7d362eaa-8130-410a-bf23-7be549177af0
https://www.ghanabusinessweb.com/accra-adabraka-health_clinics-iran_clinic-9430.html,1st Foundation Clinic,2,null,"[""internalMedicine""]",[],[],"[""Located in Dansoman, Accra, Ghana, opposite Standard Chartered Bank"",""Listed as a related place on GhanaBusinessWeb's Iran Clinic page""]",facility,cd191c26-2987-404f-b5bb-7dfe6d7a7b02,null,null,null,null,null,null,null,null,null,null,null,Opp. Standard Chartered Bank,null,null,Dansoman,null,null,Ghana,GH,null,null,null,null,clinic,null,null,null,null,null,null,9d70e24f-247c-41af-b708-cc10e99e54b1
https://www.ghanabusinessweb.com/accra-cantonments-str_0-kumoji_hospital-9464.html,1st Foundation Clinic,2,null,"[""internalMedicine""]",[],[],[],facility,6cc7060e-63f3-4e20-b83c-483ac1c3206e,null,null,null,null,null,null,null,null,null,null,null,"Opp. Standard Chartered Bank, Dansoman",null,null,Accra,null,null,Ghana,GH,null,null,null,null,clinic,null,null,null,null,null,null,97d11408-8303-44c3-83c2-72cb91c58fb1
https://www.ghanabusinessweb.com/accra-dansoman-hospitals-karikari_brobbey_hospital-9444.html,1st Foundation Clinic,2,null,"[""internalMedicine""]",[],[],[],facility,2d84a935-452c-441c-a766-8fcbb8b4e7ea,null,null,null,null,null,null,null,null,null,null,null,Opp. Standard Chartered Bank,Dansoman,null,Accra,null,null,Ghana,GH,null,null,null,null,clinic,null,null,null,null,null,null,4c17951e-87e0-4472-8cf4-189cea9782b8
https://www.ghanabusinessweb.com/accra-osu-health_clinics-gak_clinic-9409.html,1st Foundation Clinic,2,null,"[""internalMedicine""]",null,null,null,facility,78038e2e-3210-4a0b-a502-ad0f8aabbb38,null,null,null,null,null,null,null,null,null,null,null,Opp. Standard Chartered Bank,Dansoman,null,Accra,null,null,Ghana,GH,null,null,null,null,clinic,null,null,null,null,null,null,a6ec226d-a88e-4366-b390-5c709ef54e92
https://www.ghanayello.com/company/54942/2BN_Military_Hospital,2BN Military Hospital,3,627575867a35920016fe144c,"[""internalMedicine"",""dentistry"",""emergencyMedicine""

In [0]:
# Tumhari table ka exact naam
table_name = "workspace.default.virtue_foundation_ghana_v_0_3_sheet_1"

try:
    # 1. Table load karna
    raw_df = spark.table(table_name)
    
    # 2. Filtering: Sirf wo rows jahan kuch toh information ho
    # Hum 'description' aur 'capability' dono check kar rahe hain
    df_clean = raw_df.filter(
        (raw_df.description.isNotNull()) | (raw_df.capability.isNotNull())
    )
    
    print(f"✅ Step 2: Table loaded!")
    print(f"📊 Total Rows: {raw_df.count()} |  Rows (Not Null): {df_clean.count()}")
    display(df_clean.limit(3))
except Exception as e:
    print(f"❌ Error: Table is not loaded. Check name: {e}")

✅ Step 2: Table loaded!
📊 Total Rows: 995 | Kaam ki Rows (Not Null): 988


source_url,name,pk_unique_id,mongo DB,specialties,procedure,equipment,capability,organization_type,content_table_id,phone_numbers,email,websites,officialWebsite,yearEstablished,acceptsVolunteers,facebookLink,twitterLink,linkedinLink,instagramLink,logo,address_line1,address_line2,address_line3,address_city,address_stateOrRegion,address_zipOrPostcode,address_country,address_countryCode,countries,missionStatement,missionStatementLink,organizationDescription,facilityTypeId,operatorTypeId,affiliationTypeIds,description,area,numberDoctors,capacity,unique_id
https://www.linkedin.com/company/waaf/,"109/No 1 Bekwai Rd (Near Mexico Hotel) Takoradi, Ghana",1,62aa51490990af00169ab9ed,"[""infectiousDiseases"",""maternalFetalMedicineOrPerinatology"",""publicHealth"",""socialAndBehavioralSciences"",""hospiceAndPalliativeInternalMedicine"",""globalHealthAndInternationalHealth""]",null,null,"[""Has a location at 109/No 1 Bekwai Rd (Near Mexico Hotel) Takoradi, Ghana"",""Specialties include HIV, AIDS, PMTCT, Behavior Change Communication, HIV Testing and Counseling, Community Outreach, Stigma Reduction, Hospice / Home Based Care, global health, and public health""]",facility,a77400f4-6203-4b0d-84ad-fd143bd768e3,"[""+233249354576"",""+233203928883""]",null,"[""waafweb.org""]",waafweb.org,null,null,null,null,null,null,null,109/No 1 Bekwai Rd (Near Mexico Hotel),null,null,Takoradi,null,null,Ghana,GH,null,null,null,null,clinic,null,null,"WAAF is committed to battling HIV/AIDS, TB, and other conditions through grassroot initiatives throughout communities. The main objectives for WAAF are to deliver prevention, behavioral, care and support services for the general population but with emphasis on vulnerable populations. Our ultimate goal is a future where HIV and AIDS is no longer an epidemic, and people are no longer stigmatized. The organization’s vision and mission guide for daily and long-term operations, as detailed in the organization’s strategic plan document.",null,null,null,7d362eaa-8130-410a-bf23-7be549177af0
https://www.ghanabusinessweb.com/accra-adabraka-health_clinics-iran_clinic-9430.html,1st Foundation Clinic,2,null,"[""internalMedicine""]",[],[],"[""Located in Dansoman, Accra, Ghana, opposite Standard Chartered Bank"",""Listed as a related place on GhanaBusinessWeb's Iran Clinic page""]",facility,cd191c26-2987-404f-b5bb-7dfe6d7a7b02,null,null,null,null,null,null,null,null,null,null,null,Opp. Standard Chartered Bank,null,null,Dansoman,null,null,Ghana,GH,null,null,null,null,clinic,null,null,null,null,null,null,9d70e24f-247c-41af-b708-cc10e99e54b1
https://www.ghanabusinessweb.com/accra-cantonments-str_0-kumoji_hospital-9464.html,1st Foundation Clinic,2,null,"[""internalMedicine""]",[],[],[],facility,6cc7060e-63f3-4e20-b83c-483ac1c3206e,null,null,null,null,null,null,null,null,null,null,null,"Opp. Standard Chartered Bank, Dansoman",null,null,Accra,null,null,Ghana,GH,null,null,null,null,clinic,null,null,null,null,null,null,97d11408-8303-44c3-83c2-72cb91c58fb1


In [0]:
def setup_extractor(api_key):
    # Llama 3.3 70B is smart but follows strict JSON
    # llm = ChatGroq(model_name="llama-3.3-70b-versatile", 
    #                temperature=0, groq_api_key=api_key)
    

    llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0, groq_api_key=api_key)

    return llm.with_structured_output(MedicalKnowledge)

# Enter your key here
initial_key = getpass.getpass("Put Groq API Key : ")
extractor = setup_extractor(initial_key)

def analyze_medical_report(facility_name, desc, cap):
    # Fixing prompts
    combined_text = f"Facility Name: {facility_name}\nDescription: {desc}\nCapability: {cap}"
    
    prompt = f"""
    Extract medical facts for `{facility_name}` strictly following Virtue Foundation standards.

    ### CRITICAL RULES:
    1. **NO HALLUCINATION**: DO NOT list 'Oxygen', 'Generator', or 'Backup Power' unless they are explicitly mentioned in the text below. If missing, return [].
    2. NEVER return the string "null". If no data is found, return an empty list [].
    3. **JSON FORMAT**: Ensure 'is_suspicious' is a raw boolean (true/false) WITHOUT quotes. 
    4. **KEY UNICITY**: NEVER repeat a key (like 'equipment') in the output.
    5.'procedures', 'equipment', and 'capability' MUST ALWAYS be JSON arrays [].

    ### CATEGORY DEFINITIONS:
    - 'equipment': Imaging machines, lab tools, and CRITICAL UTILITIES (Piped Oxygen, Oxygen Plants, Generators).
    - 'capability': Specialized units (ICU, NICU, Trauma), clinical programs. (EXCLUDE: Address, Phone, Email, URLs).

    ### AUDIT LOGIC:
    - Set 'is_suspicious' to true ONLY IF:
      (a) Facility claims to be a 'Hospital', 'Surgery', 'Theatre', or 'Centre'.
      (b) AND 'Oxygen' or 'Generator' is NOT mentioned in the text.

    ### DATA TO ANALYZE:
    Facility Name: {facility_name}
    Description: {desc}
    Capability Info: {cap}
    """    
    # AI ko call karna
    return extractor.invoke(prompt)

print("✅ Step 3: AI Logic Ready!")

Pehli Groq API Key daalo:  [REDACTED]

✅ Step 3: AI Logic Ready!


In [0]:
all_extracted_data = []
# Trying with 50 records
pdf_data = df_clean.limit(50).toPandas() 

print("🚀 Processing starts...")

for index, row in pdf_data.iterrows():
    success = False
    while not success:
        try:
            # Data columns uthana
            f_name = row.get('name', 'Unknown')
            f_desc = row.get('description', '')
            f_cap = row.get('capability', '')
            
            # AI Extraction
            result = analyze_medical_report(f_name, f_desc, f_cap)
            
            # Save results
            res_dict = result.model_dump() # Updated for Pydantic V2

            # 1. SPECIALTY MAPPING (Official ZIP Rules fix)
            # Allowed list based on your schema
            allowed_list = ["internalMedicine", "familyMedicine", "pediatrics", "cardiology", "generalSurgery", "emergencyMedicine", "gynecologyAndObstetrics", "orthopedicSurgery", "dentistry", "ophthalmology"]
            
            current_spec = str(res_dict.get('specialty', '')).lower()
            
            # Manual Mapping logic
            if any(word in current_spec for word in ['psychiatry', 'mental', 'generalmedicine']):
                res_dict['specialty'] = "internalMedicine"
            elif any(word in current_spec for word in ['therapy', 'rehab', 'clinic']) or not res_dict.get('specialty'):
                res_dict['specialty'] = "familyMedicine"
            elif any(word in current_spec for word in ['maternity', 'women']):
                res_dict['specialty'] = "gynecologyAndObstetrics"
            
            # Final Safety: 
            if res_dict['specialty'] not in allowed_list:
                res_dict['specialty'] = "familyMedicine"
            
            # 1. MANUAL AUDIT OVERRIDE ( AI hallucinate fix)
            hosp_keywords = ['hospital', 'surgery', 'theatre', 'medical centre', 'centre']
            is_large_facility = any(k in f_name.lower() for k in hosp_keywords)
            
            # Check if actual infrastructure is mentioned in the input text (not just AI output)
            raw_text = (f_desc + " " + f_cap).lower()
            has_infrastructure = any(infra in raw_text for infra in ['oxygen', 'generator', 'backup power', 'plant'])

            if is_large_facility and not has_infrastructure:
                res_dict['is_suspicious'] = True
                res_dict['suspicion_reason'] = "Facility claims to be a large medical center/hospital but provides no evidence of critical infrastructure (Oxygen/Power)."
            else:
                # if suspicious else reason empty
                if not res_dict.get('is_suspicious'):
                    res_dict['is_suspicious'] = False
                    res_dict['suspicion_reason'] = ""

            # 2. ADDRESS CLEANUP (Extra safety layer)
            banned = ['located', 'road', 'rd', 'street', 'phone', 'tel', '+233', 'email', 'website', 'http']
            res_dict['capability'] = [c for c in res_dict.get('capability', []) if not any(w in c.lower() for w in banned)]
            # 4. FINAL CLEANUP: Ensure no field is None/NaN
            for field in ['procedures', 'equipment', 'capability']:
                if res_dict.get(field) is None: res_dict[field] = []
            if res_dict.get('suspicion_reason') is None: res_dict['suspicion_reason'] = ""

            
            #logic end
            
            res_dict['unique_id'] = row.get('unique_id', index)
            all_extracted_data.append(res_dict)
            
            print(f"✅ Row {index} Done")
            success = True
            time.sleep(2) # API safety

        except Exception as e:
            if "429" in str(e):
                print(f"\n⚠️ LIMIT EXHAUSTED! Account change karo.")
                new_key = getpass.getpass("Nayi Groq Key dalo (New Gmail Account): ")
                extractor = setup_extractor(new_key)
                print("🔄 Key Updated. Retrying same row...")
                time.sleep(3)
            else:
                print(f"❌ Error at row {index}: {e}")
                success = True # Error skip 

# Final Table Display
if all_extracted_data:
    final_report = pd.DataFrame(all_extracted_data)
    display(final_report)
    # Excel download ke liye CSV save kar lo
    final_report.to_csv("Virtue_Final_Audit.csv", index=False)
    print("📂 Results saved to Virtue_Final_Audit.csv")

🚀 Processing starts...
✅ Row 0 Done
✅ Row 1 Done
✅ Row 2 Done
✅ Row 3 Done
✅ Row 4 Done
✅ Row 5 Done


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
all_extracted_data = []
# texting with all  records
pdf_data = df_clean.toPandas() 

print("🚀 Processing starts...")

for index, row in pdf_data.iterrows():
    success = False
    while not success:
        try:
            # Data columns uthana
            f_name = row.get('name', 'Unknown')
            f_desc = row.get('description', '')
            f_cap = row.get('capability', '')
            
            # AI Extraction
            result = analyze_medical_report(f_name, f_desc, f_cap)
            
            # Save results
            res_dict = result.model_dump() # Updated for Pydantic V2

            # 1. SPECIALTY MAPPING (Official ZIP Rules fix)
            # Allowed list based on your schema
            allowed_list = ["internalMedicine", "familyMedicine", "pediatrics", "cardiology", "generalSurgery", "emergencyMedicine", "gynecologyAndObstetrics", "orthopedicSurgery", "dentistry", "ophthalmology"]
            
            current_spec = str(res_dict.get('specialty', '')).lower()
            
            # Manual Mapping logic
            if any(word in current_spec for word in ['psychiatry', 'mental', 'generalmedicine']):
                res_dict['specialty'] = "internalMedicine"
            elif any(word in current_spec for word in ['therapy', 'rehab', 'clinic']) or not res_dict.get('specialty'):
                res_dict['specialty'] = "familyMedicine"
            elif any(word in current_spec for word in ['maternity', 'women']):
                res_dict['specialty'] = "gynecologyAndObstetrics"
            
            # Final Safety: 
            if res_dict['specialty'] not in allowed_list:
                res_dict['specialty'] = "familyMedicine"
            
            # 1. MANUAL AUDIT OVERRIDE (if AI hallucinate )
            hosp_keywords = ['hospital', 'surgery', 'theatre', 'medical centre', 'centre']
            is_large_facility = any(k in f_name.lower() for k in hosp_keywords)
            
            # Check if actual infrastructure is mentioned in the input text (not just AI output)
            raw_text = (f_desc + " " + f_cap).lower()
            has_infrastructure = any(infra in raw_text for infra in ['oxygen', 'generator', 'backup power', 'plant'])

            if is_large_facility and not has_infrastructure:
                res_dict['is_suspicious'] = True
                res_dict['suspicion_reason'] = "Facility claims to be a large medical center/hospital but provides no evidence of critical infrastructure (Oxygen/Power)."
            else:
                # if suspicious else reason empty
                if not res_dict.get('is_suspicious'):
                    res_dict['is_suspicious'] = False
                    res_dict['suspicion_reason'] = ""

            # 2. ADDRESS CLEANUP (Extra safety layer)
            banned = ['located', 'road', 'rd', 'street', 'phone', 'tel', '+233', 'email', 'website', 'http']
            res_dict['capability'] = [c for c in res_dict.get('capability', []) if not any(w in c.lower() for w in banned)]
            # 4. FINAL CLEANUP: Ensure no field is None/NaN
            for field in ['procedures', 'equipment', 'capability']:
                if res_dict.get(field) is None: res_dict[field] = []
            if res_dict.get('suspicion_reason') is None: res_dict['suspicion_reason'] = ""

            
            #logic end
            
            res_dict['unique_id'] = row.get('unique_id', index)
            all_extracted_data.append(res_dict)
            
            print(f"✅ Row {index} Done")
            success = True
            time.sleep(2) # API safety

        except Exception as e:
            if "429" in str(e):
                print(f"\n⚠️ LIMIT EXHAUSTED! Change Account.")
                new_key = getpass.getpass("Put new Groq Key  (New Gmail Account): ")
                extractor = setup_extractor(new_key)
                print("🔄 Key Updated. Retrying same row...")
                time.sleep(3)
            else:
                print(f"❌ Error at row {index}: {e}")
                success = True # Error skip 

# Final Table Display
if all_extracted_data:
    final_report = pd.DataFrame(all_extracted_data)
    display(final_report)
    # Excel download ke liye CSV save kar lo
    final_report.to_csv("Virtue_Final_Audit.csv", index=False)
    print("📂 Results saved to Virtue_Final_Audit.csv")

🚀 Processing starts...
✅ Row 0 Done
✅ Row 1 Done
✅ Row 2 Done
✅ Row 3 Done
✅ Row 4 Done
✅ Row 5 Done
✅ Row 6 Done
✅ Row 7 Done
✅ Row 8 Done
✅ Row 9 Done
✅ Row 10 Done
✅ Row 11 Done
✅ Row 12 Done
✅ Row 13 Done
✅ Row 14 Done
✅ Row 15 Done
✅ Row 16 Done
✅ Row 17 Done
✅ Row 18 Done
✅ Row 19 Done
✅ Row 20 Done
✅ Row 21 Done
✅ Row 22 Done
✅ Row 23 Done
✅ Row 24 Done
✅ Row 25 Done
✅ Row 26 Done
✅ Row 27 Done
✅ Row 28 Done
✅ Row 29 Done
✅ Row 30 Done
✅ Row 31 Done
✅ Row 32 Done
✅ Row 33 Done
✅ Row 34 Done
✅ Row 35 Done
✅ Row 36 Done
✅ Row 37 Done
✅ Row 38 Done
✅ Row 39 Done
✅ Row 40 Done
✅ Row 41 Done
✅ Row 42 Done
✅ Row 43 Done
✅ Row 44 Done
✅ Row 45 Done
✅ Row 46 Done
✅ Row 47 Done
✅ Row 48 Done
✅ Row 49 Done
✅ Row 50 Done
✅ Row 51 Done
✅ Row 52 Done
✅ Row 53 Done
✅ Row 54 Done
✅ Row 55 Done
✅ Row 56 Done
✅ Row 57 Done
✅ Row 58 Done
✅ Row 59 Done
✅ Row 60 Done
✅ Row 61 Done
✅ Row 62 Done
✅ Row 63 Done
✅ Row 64 Done
✅ Row 65 Done
✅ Row 66 Done
✅ Row 67 Done
✅ Row 68 Done
✅ Row 69 Done
✅ Row 7

Nayi Groq Key dalo (New Gmail Account):  [REDACTED]

🔄 Key Updated. Retrying same row...
✅ Row 102 Done
✅ Row 103 Done
✅ Row 104 Done
✅ Row 105 Done
✅ Row 106 Done
✅ Row 107 Done
✅ Row 108 Done
✅ Row 109 Done
✅ Row 110 Done
✅ Row 111 Done
✅ Row 112 Done
✅ Row 113 Done
✅ Row 114 Done
✅ Row 115 Done
✅ Row 116 Done
✅ Row 117 Done
❌ Error at row 118: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=MedicalKnowledge> {"facility_name": "Ayeduase Health Centre", "specialty": "", "procedures": [], "equipment": [], "capability": ["Located in Ayeduase-Kumasi, Kumasi, Ghana", "Always open", "Aims to promote good health and ensure access to quality and affordable healthcare."], "is_suspicious": false, "suspicion_reason": "", "capability_info": ["Located in Ayeduase-Kumasi, Kumasi, Ghana", "Always open", "Aims to promote good health and ensure access to quality and afford

Nayi Groq Key dalo (New Gmail Account):  [REDACTED]

🔄 Key Updated. Retrying same row...
✅ Row 148 Done
✅ Row 149 Done
✅ Row 150 Done
✅ Row 151 Done
✅ Row 152 Done
✅ Row 153 Done
✅ Row 154 Done
✅ Row 155 Done
✅ Row 156 Done
✅ Row 157 Done
✅ Row 158 Done
✅ Row 159 Done
✅ Row 160 Done
✅ Row 161 Done
✅ Row 162 Done
✅ Row 163 Done
✅ Row 164 Done
✅ Row 165 Done
✅ Row 166 Done
✅ Row 167 Done
✅ Row 168 Done
✅ Row 169 Done
✅ Row 170 Done
✅ Row 171 Done
✅ Row 172 Done
✅ Row 173 Done
✅ Row 174 Done
✅ Row 175 Done
✅ Row 176 Done
✅ Row 177 Done
✅ Row 178 Done
✅ Row 179 Done
✅ Row 180 Done
✅ Row 181 Done
✅ Row 182 Done
✅ Row 183 Done
✅ Row 184 Done
✅ Row 185 Done
✅ Row 186 Done
✅ Row 187 Done
✅ Row 188 Done
✅ Row 189 Done
✅ Row 190 Done
✅ Row 191 Done
✅ Row 192 Done
✅ Row 193 Done
✅ Row 194 Done
✅ Row 195 Done
✅ Row 196 Done
✅ Row 197 Done
✅ Row 198 Done
✅ Row 199 Done
✅ Row 200 Done
✅ Row 201 Done
✅ Row 202 Done
✅ Row 203 Done
✅ Row 204 Done
✅ Row 205 Done
✅ Row 206 Done
✅ Row 207 Done
✅ Row 208 Done
✅ Row 209 Done
✅ Row 210 Done
✅ Row 211 Done
✅ Ro

...

🔄 Key Updated. Retrying same row...
✅ Row 838 Done
✅ Row 839 Done
✅ Row 840 Done
✅ Row 841 Done
✅ Row 842 Done
✅ Row 843 Done

⚠️ LIMIT EXHAUSTED! Account change karo.


Nayi Groq Key dalo (New Gmail Account):  [REDACTED]

🔄 Key Updated. Retrying same row...
✅ Row 844 Done
✅ Row 845 Done
✅ Row 846 Done
✅ Row 847 Done
✅ Row 848 Done
✅ Row 849 Done
✅ Row 850 Done
✅ Row 851 Done
✅ Row 852 Done
✅ Row 853 Done
✅ Row 854 Done
✅ Row 855 Done
✅ Row 856 Done
✅ Row 857 Done
✅ Row 858 Done
✅ Row 859 Done
✅ Row 860 Done
✅ Row 861 Done
✅ Row 862 Done
✅ Row 863 Done
✅ Row 864 Done
❌ Error at row 865: unsupported operand type(s) for +: 'NoneType' and 'str'
❌ Error at row 866: unsupported operand type(s) for +: 'NoneType' and 'str'
✅ Row 867 Done
✅ Row 868 Done
✅ Row 869 Done
✅ Row 870 Done
✅ Row 871 Done
✅ Row 872 Done
✅ Row 873 Done
✅ Row 874 Done
✅ Row 875 Done
✅ Row 876 Done
✅ Row 877 Done
✅ Row 878 Done
✅ Row 879 Done
✅ Row 880 Done
✅ Row 881 Done
✅ Row 882 Done
✅ Row 883 Done
✅ Row 884 Done
✅ Row 885 Done
✅ Row 886 Done
✅ Row 887 Done
✅ Row 888 Done
✅ Row 889 Done
✅ Row 890 Done
✅ Row 891 Done
✅ Row 892 Done
✅ Row 893 Done
✅ Row 894 Done
✅ Row 895 Done
✅ Row 896 Done
✅ Row 897 Done
✅ Row 898 Done
✅ Row 899 Done
✅ 

facility_name,specialty,procedures,equipment,capability,is_suspicious,suspicion_reason,unique_id
"109/No 1 Bekwai Rd (Near Mexico Hotel) Takoradi, Ghana",internalMedicine,"List(HIV Testing and Counseling, Community Outreach, Stigma Reduction, Hospice / Home Based Care)",List(),"List(Specialties include HIV, AIDS, PMTCT, Behavior Change Communication, global health, and public health)",false,,7d362eaa-8130-410a-bf23-7be549177af0
1st Foundation Clinic,familyMedicine,List(),List(),List(),false,,9d70e24f-247c-41af-b708-cc10e99e54b1
1st Foundation Clinic,internalMedicine,List(),List(),List(),false,,97d11408-8303-44c3-83c2-72cb91c58fb1
1st Foundation Clinic,internalMedicine,List(),List(),List(),false,,4c17951e-87e0-4472-8cf4-189cea9782b8
1st Foundation Clinic,familyMedicine,List(),List(),List(),false,,a6ec226d-a88e-4366-b390-5c709ef54e92
2BN Military Hospital,internalMedicine,List(),List(),List(),true,Facility claims to be a large medical center/hospital but provides no evidence of critical infrastructure (Oxygen/Power).,8944be75-1ef2-49f7-84a9-763dc4000c7b
37 Military Hospital,internalMedicine,List(),List(),List(),true,Facility claims to be a large medical center/hospital but provides no evidence of critical infrastructure (Oxygen/Power).,3fef3d66-36d4-4c8c-b132-209d75eb28c5
37 Military Hospital,internalMedicine,List(),List(),List(),true,Facility claims to be a large medical center/hospital but provides no evidence of critical infrastructure (Oxygen/Power).,f9d8de8a-10e1-4101-afad-6c0917fc0233
"3E Medical Center - Accra, Ghana",internalMedicine,List(),List(),List(),false,,438f3e65-d688-4db6-bc9f-0d2f6edbafe6
3Way Family Care Clinic,familyMedicine,List(),List(),List(),false,,d9051330-1480-4c71-a0ba-365d0d586e40


📂 Results saved to Virtue_Final_Audit.csv
